In [1]:
# CELL 1: Install smolagents and wake up your first AI Agent with a Calculator Hand
!pip install -q "smolagents[transformers]" accelerate pydantic pandas tabulate

import os
import gc
import torch
from kaggle_secrets import UserSecretsClient
from smolagents import CodeAgent, InferenceClientModel, TransformersModel, tool

# 1. Unlock our HF_TOKEN from Kaggle's safe
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

# 2. Build the "Calculator Hand" using the @tool sticker, Type Hints, and a Docstring
@tool
def calculate_inference_cost(input_tokens: int, output_tokens: int, cost_per_million: float) -> float:
    """Calculates the USD cost of an LLM run given token counts.

    Args:
        input_tokens: Total prompt tokens sent to the model.
        output_tokens: Total completion tokens generated by the model.
        cost_per_million: Price in USD per 1,000,000 tokens.
    """
    return round(((input_tokens + output_tokens) / 1_000_000) * cost_per_million, 6)

# 3. Connect the Brain (Tries free cloud API first; if busy, runs directly on your Kaggle T4 GPU!)
try:
    brain = InferenceClientModel(model_id="Qwen/Qwen2.5-Coder-32B-Instruct")
    agent = CodeAgent(tools=[calculate_inference_cost], model=brain)
    agent.run(
        "If my AI system runs 25,000 queries a day with 1,800 input tokens and 350 output tokens "
        "each at $0.40 per million tokens, what is my total 30-day monthly cost in USD?"
    )
except Exception as e:
    print("Cloud API busy, switching to our own Kaggle T4 GPU brain...")
    local_brain = TransformersModel(
        model_id="Qwen/Qwen2.5-3B-Instruct",
        device_map="cuda:0",
        torch_dtype="float16"
    )
    agent = CodeAgent(tools=[calculate_inference_cost], model=local_brain)
    agent.run(
        "If my AI system runs 25,000 queries a day with 1,800 input tokens and 350 output tokens "
        "each at $0.40 per million tokens, what is my total 30-day monthly cost in USD?"
    )
    del local_brain
    gc.collect()
    torch.cuda.empty_cache()

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ If my AI system runs 25,000 queries a day with 1,800 input tokens and 350 output tokens each at $0.40 per       │
│ million tokens, what is my total 30-day monthly cost in USD?                                                    │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-Coder-32B-Instruct ────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Error in generating model output:
(Request ID: Root=1-6af1a287-5752feb0344005ca601d8fb8;61896fea-c024-4b97-847a-8637b6bdeab3)

Bad request:
{'message': "The requested model 'Qwen/Qwen2.5-Coder-32B-Instruct' is not supported by any provider you have 
enabled.", 'type': 'invalid_request_error', 'param': 'model', 'code': 'model_not_supported'}

[Step 1: Duration 0.28 seconds]

Cloud API busy, switching to our own Kaggle T4 GPU brain...


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

... (11 outputs left out)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  total_input_tokens_per_day = 25000 * 1800                                                                        
  total_output_tokens_per_day = 25000 * 350                                                                        
  total_tokens_per_day = total_input_tokens_per_day + total_output_tokens_per_day                                  
                                                                                                                   
  total_tokens_per_month = total_tokens_per_day * 30                                                               
                                                                                                                   
  cost_per_million_tokens = 0.40                                                                                   
  total_cost_per_month = (total_tokens_per_month / 1000000) * cost_per_million_tokens                              
                                                                                                                   
  final_answer(total_cost_per_month)                                                                               
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: 645.0

[Step 1: Duration 14.54 seconds| Input tokens: 2,171 | Output tokens: 174]

In [2]:
# CELL 2: The T4 GPU Speed, Memory & Strict JSON Accuracy Race
import gc
import json
import re
import time
import torch
import pandas as pd
from pydantic import BaseModel, Field, ValidationError
from transformers import AutoModelForCausalLM, AutoTokenizer

# 1. Make sure our NVIDIA T4 GPU is awake
assert torch.cuda.is_available(), "GPU is OFF! Open Kaggle Right Sidebar -> Accelerator -> Select GPU T4 x2"
print(f"GPU Ready: {torch.cuda.get_device_name(0)}")

# 2. The Pydantic Border Guard: Every AI answer MUST have these exact 4 fields and types
class InvoiceTicket(BaseModel):
    vendor_name: str = Field(description="Company sending the bill")
    invoice_number: str = Field(description="Invoice ID code")
    total_amount_usd: float = Field(description="Dollar amount as a decimal")
    is_urgent: bool = Field(description="True if urgent or due within 7 days, else False")

# 3. Five messy human emails to test our two AI brains
TEST_SAMPLES = [
    "Hey finance team, CloudScale AWS Services just sent over bill #INV-9021 for $1,420.50. It's due tomorrow so please rush it!",
    "Attached is the monthly statement from Apex Legal Partners (Ref: ALP-2026-88). Total balance is $850.00, payable in 30 days.",
    "URGENT: Server hosting invoice #HK-4410 from Hetzner Online GmbH for $64.25 will auto-suspend in 48 hours if unpaid.",
    "Standard office catering receipt from GreenLeaf Foods, invoice ID GL-772, coming out to $215.75. Net-45 payment terms.",
    "Final notice from Datadog Monitoring Inc. regarding invoice DD-10994 for $3,100.00. Immediate settlement required today."
]

SYSTEM_PROMPT = """You are a strict data-extraction engine.
Extract the invoice details from the user's text and return ONLY a valid JSON object with these exact 4 keys:
{
  "vendor_name": "string",
  "invoice_number": "string",
  "total_amount_usd": float,
  "is_urgent": boolean
}
Do not include explanations or extra text."""

def clean_and_validate_json(raw_text: str) -> bool:
    """Strips markdown fences and checks if the output passes our Pydantic border guard."""
    try:
        cleaned = re.sub(r"```json\s*|\s*```", "", raw_text).strip()
        parsed_dict = json.loads(cleaned)
        InvoiceTicket.model_validate(parsed_dict)
        return True
    except (json.JSONDecodeError, ValidationError, TypeError):
        return False

# 4. Run the race between Driver A (1.5B) and Driver B (3B)
MODELS_TO_TEST = ["Qwen/Qwen2.5-1.5B-Instruct", "Qwen/Qwen2.5-3B-Instruct"]
benchmark_results = []

for model_id in MODELS_TO_TEST:
    print(f"\n{'='*55}\nRacing Model: {model_id}\n{'='*55}")
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(device="cuda:0")
    
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        device_map="cuda:0"
    )
    
    total_tokens = 0
    total_seconds = 0.0
    valid_json_count = 0
    
    for idx, sample_text in enumerate(TEST_SAMPLES, start=1):
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": sample_text}
        ]
        prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda:0")
        input_len = inputs.input_ids.shape[1]
        
        torch.cuda.synchronize()
        start_t = time.perf_counter()
        
        with torch.no_grad():
            output_ids = model.generate(**inputs, max_new_tokens=128, do_sample=False)
            
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - start_t
        
        new_tokens = output_ids[0][input_len:]
        generated_text = tokenizer.decode(new_tokens, skip_special_tokens=True)
        
        is_valid = clean_and_validate_json(generated_text)
        if is_valid:
            valid_json_count += 1
            
        total_tokens += len(new_tokens)
        total_seconds += elapsed
        print(f"  Email {idx}/5 | Passed Border Guard: {is_valid} | Speed: {elapsed:.2f}s")
        
    peak_vram_gb = torch.cuda.max_memory_allocated(device="cuda:0") / (1024 ** 3)
    tok_per_sec = total_tokens / total_seconds
    accuracy_pct = (valid_json_count / len(TEST_SAMPLES)) * 100
    
    benchmark_results.append({
        "Model Name": model_id,
        "Parameters": "1.5B" if "1.5B" in model_id else "3.0B",
        "Peak VRAM (GB)": round(peak_vram_gb, 2),
        "Throughput (tok/s)": round(tok_per_sec, 1),
        "JSON Validity (%)": f"{accuracy_pct:.0f}%"
    })
    
    # Janitor sweeps the GPU desk clean before the next model!
    del model
    del tokenizer
    gc.collect()
    torch.cuda.empty_cache()

df_results = pd.DataFrame(benchmark_results)
df_results.to_csv("day1_gpu_benchmarks.csv", index=False)

print("\n=== YOUR FINAL RACE SCOREBOARD ===\n")
print(df_results.to_markdown(index=False))

GPU Ready: Tesla T4

Racing Model: Qwen/Qwen2.5-1.5B-Instruct


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (3 outputs left out)

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

  Email 1/5 | Passed Border Guard: True | Speed: 1.93s
  Email 2/5 | Passed Border Guard: True | Speed: 2.00s
  Email 3/5 | Passed Border Guard: True | Speed: 1.88s
  Email 4/5 | Passed Border Guard: True | Speed: 1.84s
  Email 5/5 | Passed Border Guard: True | Speed: 2.05s

Racing Model: Qwen/Qwen2.5-3B-Instruct


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

  Email 1/5 | Passed Border Guard: True | Speed: 2.46s
  Email 2/5 | Passed Border Guard: True | Speed: 2.56s
  Email 3/5 | Passed Border Guard: True | Speed: 2.45s
  Email 4/5 | Passed Border Guard: True | Speed: 2.32s
  Email 5/5 | Passed Border Guard: True | Speed: 2.50s

=== YOUR FINAL RACE SCOREBOARD ===

| Model Name                 | Parameters   |   Peak VRAM (GB) |   Throughput (tok/s) | JSON Validity (%)   |
|:---------------------------|:-------------|-----------------:|---------------------:|:--------------------|
| Qwen/Qwen2.5-1.5B-Instruct | 1.5B         |             8.64 |                 26.1 | 100%                |
| Qwen/Qwen2.5-3B-Instruct   | 3.0B         |            11.52 |                 20.6 | 100%                |
